In [86]:
import pandas as pd
import requests
import numpy as np
import math
import os

In [87]:
#url = http://localhost:8080/ors/v2/directions/driving-car?start=-37.81020316,145.0355016&end=-37.911108,144.993524


In [88]:
"""
ORS_BASE_URL = "http://localhost:8080/ors/v2/directions/driving-car"
first_point = (-37.81020316, 145.0355016)
second_point = (-37.911108, 144.993524)
url = f'{ORS_BASE_URL}?start={first_point[0]},{first_point[1]}&end={second_point[0]},{second_point[1]}'

response = requests.get(url)

json_response = response.json()
json_response
"""

'\nORS_BASE_URL = "http://localhost:8080/ors/v2/directions/driving-car"\nfirst_point = (-37.81020316, 145.0355016)\nsecond_point = (-37.911108, 144.993524)\nurl = f\'{ORS_BASE_URL}?start={first_point[0]},{first_point[1]}&end={second_point[0]},{second_point[1]}\'\n\nresponse = requests.get(url)\n\njson_response = response.json()\njson_response\n'

# Distance to primary and secondary schools


In [89]:
# read school and rental data
rental_data = pd.read_csv("../data/curated/rental_data_sf.csv")
school_data_s2 = pd.read_csv("../data/curated/school_data/school_data_s2.csv")
rental_data["coordinates"] = rental_data["coordinates"].apply(eval)
school_data_s2["coordinate"] = school_data_s2["coordinate"].apply(eval)
rental_data["SA2_CODE21"].value_counts()

SA2_CODE21
206041504.0    299
206041505.0    248
206041509.0    234
206041503.0    161
206041118.0    138
              ... 
201031014.0      1
211051283.0      1
212041316.0      1
216011409.0      1
206041120.0      1
Name: count, Length: 496, dtype: int64

In [92]:
EARTH_RADIUS = 6371
def calculate_distance(coordinate1, coordinate2):
    """ Function calculates the haversine distance between two coordinates """
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(coordinate1[0])
    lon1 = np.radians(coordinate1[1])
    lat2 = np.radians(coordinate2[0])
    lon2 = np.radians(coordinate2[1])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = EARTH_RADIUS * c
    
    return distance
    
    
    

In [97]:
def distance_closest_school(rental_data_coordinate, rental_data_sa2, school_data, new_col_names, school_types):
    """ Function finds the distance to the closest school from the rental property. If new_col_names list is of length 
    two it indicates that we are searching for closest secondary school and the function returns both distance to closest school 
    and boolean indicating if school is good. If new_col_names list is of length one, it indicates we are searching for primary schools and
    we only return distance to school """
    
    distance_dict = {}
    # Find closest school in SA2 area
    for index, school in school_data[(school_data["sa2_code_2021"] == rental_data_sa2) 
                                     & (school_data["school_type"].isin(school_types))].iterrows():
        distance_dict[school["coordinate"]] = calculate_distance(rental_data_coordinate, school["coordinate"])

    if distance_dict:
        # return school with shortest distance
        school_distance = min(distance_dict.items(), key=lambda x: x[1])[1]
        school_coordinate = min(distance_dict.items(), key=lambda x: x[1])[0]
        
        # for primary school just return distance to closest school
        if len(new_col_names) == 1:
            return pd.Series({new_col_names[0]: school_distance})
        # for secondary school return distance to closest school and if the school is top vce school
        elif len(new_col_names) == 2:
            # is school a top school
            top_school = ~(school_data[(school_data["coordinate"] == school_coordinate) & 
                                       (school_data["sa2_code_2021"] == rental_data_sa2)]["2022 ranking"].isna()).iloc[0]
            return pd.Series({new_col_names[0]: school_distance, 
                              new_col_names[1]: top_school})
    
    # school not found in SA2 area
    else:
        # Find closest school by comparing with all schools in victoria
        for index, school in school_data[school_data["school_type"].isin(school_types)].iterrows():
            distance_dict[school["coordinate"]] = [
            calculate_distance(rental_data_coordinate, school["coordinate"]), school["sa2_code_2021"]]
       
        if distance_dict:
            # return school with shortest distance 
            school_distance = min(distance_dict.items(), key=lambda x: x[1][0])[1][0]
            school_sa2_code_2021 = min(distance_dict.items(), key=lambda x: x[1][0])[1][1]
            school_coordinate = min(distance_dict.items(), key=lambda x: x[1][0])[0]
            # for primary school just return distance to closest school
            if len(new_col_names) == 1:
                return pd.Series({new_col_names[0]: school_distance})
            # for secondary school return distance to closest school and if the school is top vce school
            elif len(new_col_names) == 2:
                # is school a top school
                top_school = ~(school_data[(school_data["coordinate"] == school_coordinate) & 
                                       (school_data["sa2_code_2021"] == school_sa2_code_2021)]["2022 ranking"].isna()).iloc[0]
                return pd.Series({new_col_names[0]: school_distance, 
                                new_col_names[1]: top_school})
        
# find closest primary school    
primary_school = ["primary", "pri/sec"]
new_col_names_primary = ["distance primary school"]
new_columns_primary = rental_data.apply(
    lambda x: distance_closest_school(x["coordinates"], x["SA2_CODE21"], school_data_s2, new_col_names_primary, primary_school), axis=1)
rental_data = pd.concat([rental_data, new_columns_primary], axis=1)

# find closest secondary school
secondary_school = ["secondary"]
new_col_names_secondary = ["distance secondary school", "is closest secondary school good"]
new_columns_secondary = rental_data.apply(
    lambda x: distance_closest_school(x["coordinates"], x["SA2_CODE21"], school_data_s2, new_col_names_secondary, secondary_school), axis=1)
rental_data = pd.concat([rental_data, new_columns_secondary], axis=1)

7661    our lady of sion college
Name: school_name, dtype: object
557    dromana secondary college
558    dromana secondary college
Name: school_name, dtype: object
4554    mazenod college
4555    mazenod college
Name: school_name, dtype: object
1332    melbourne high
1333    melbourne high
1334    melbourne high
1335    melbourne high
Name: school_name, dtype: object
6334    ashwood high
6335    ashwood high
6336    ashwood high
6337    ashwood high
Name: school_name, dtype: object
7894    salesian college
7895    salesian college
Name: school_name, dtype: object
6334    ashwood high
6335    ashwood high
6336    ashwood high
6337    ashwood high
Name: school_name, dtype: object
6334    ashwood high
6335    ashwood high
6336    ashwood high
6337    ashwood high
Name: school_name, dtype: object
7661    our lady of sion college
Name: school_name, dtype: object
7661    our lady of sion college
Name: school_name, dtype: object
7661    our lady of sion college
Name: school_name, dtype: obje

# Find closest University Campus

In [99]:
university_sa2 = pd.read_csv("../data/curated/school_data/university_location_curated_s2.csv")
university_sa2["coordinate"] = university_sa2["coordinate"].apply(eval)

In [101]:
def distance_closest_university(rental_data_coordinate, rental_data_sa2, new_col_names, university_data):
    """ Function finds the distance to the closest university from the rental property """

    distance_dict = {}
    # Find closest university in SA2 area
    for index, university in university_data[university_data["sa2_code_2021"] == rental_data_sa2].iterrows():
        university_campus_name = university["institution"] + "-" + university["campus_name"]
        distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])
   
    # check with universities without SA2 code
    for index, university in university_data[university_data["sa2_code_2021"].isna()].iterrows():
        university_campus_name = university["institution"] + "-" + university["campus_name"]
        distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])


    if distance_dict:
        # return closest campus
        return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                          new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})
    
    else:
        # Find closest university by comparing witha all universities in victoria
        for index, university in university_data.iterrows():
            university_campus_name = university["institution"] + "-" + university["campus_name"]
            distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])
        if distance_dict:
            # return closest campus
            return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                            new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})

In [102]:
new_col_names = ["closest university campus", "distance university campus"]
new_col_university = rental_data.apply(
    lambda x: distance_closest_university(x["coordinates"], x["SA2_CODE21"], new_col_names, university_sa2), axis=1)
rental_data = pd.concat([rental_data, new_col_university], axis=1)

In [103]:
rental_data

,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,...,Weekly Rent Amount,Postcode,geometry,SA2_CODE21,SA2_NAME21,distance primary school,distance secondary school,is closest school good,closest university campus,distance university campus
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...","['Furnished*', 'Broadband internet access*', '...",...,120,3127,POINT (145.1114189 -37.8353315),207031167.0,Surrey Hills (East) - Mont Albert,0.586504,2.491226,True,monash university-notting hill,7.430317
1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",...,130,3216,POINT (144.3060777 -38.1962932),203021042.0,Highton,1.372612,2.438191,False,deakin university-waurn ponds,0.675843
2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,[],...,140,3936,POINT (145.0045603 -38.3138012),214021377.0,Dromana,4.858273,3.301447,True,monash university-notting hill,47.278807
3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,"['Built in wardrobes', 'Internal Laundry', 'Ga...",...,140,3216,POINT (144.3154857 -38.19508),203021042.0,Highton,1.114181,1.998214,False,deakin university-waurn ponds,1.502495
4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",...,150,3216,POINT (144.3107906 -38.1845196),203021042.0,Highton,0.215009,1.989564,False,deakin university-waurn ponds,1.862509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11404,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...","['Study', 'Swimming Pool', 'Ducted Cooling', '...",...,4400,3186,POINT (144.9997396 -37.9044002),208011169.0,Brighton (Vic.),0.311431,0.891118,True,monash university-notting hill,11.507926
11405,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",...,4700,3223,POINT (144.7147899 -38.1439923),203031051.0,Portarlington,3.095969,14.695112,False,the university of melbourne-parkville,44.215772
11406,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",[],...,5000,3006,POINT (144.9646982 -37.8213342),206041508.0,Southbank (West) - South Wharf,1.327110,0.766691,False,the university of melbourne-southbank,0.615756
11407,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,"['Air conditioning*', 'Built in wardrobes*', '...",...,9950,3103,POINT (145.0775507 -37.8117577),207011147.0,Balwyn,0.414506,1.204451,True,the university of melbourne-parkville,10.386190


In [104]:
rental_data.to_csv("../data/curated/rental_data_distance_uni_school_sf.csv", index=False)